hₜ = tanh(Wₓ·xₜ + W_h·hₜ₋₁ + b)

In [1]:
import torch

In [2]:
torch.manual_seed(0)

# ---------------------------------------------------------
# 1. BOYUTLARI TANIMLAMA
# ---------------------------------------------------------

n, d_in, d_h = 5, 4, 3
# n = 5     : Girdi dizisinde toplam 5 token (zaman adımı) var.
#             Örneğin: "The cat sat on mat"
#
# d_in = 4  : Her token, modele 4 sayılık bir vektör olarak veriliyor.
#             Bu vektörler kelime embedding'leri veya özellik vektörleri olabilir.
#
# d_h = 3   : RNN'in gizli durum (hidden state) vektörünün boyutu 3.
#             Model her zaman adımında geçmiş bilgiyi 3 sayılık bir vektörde tutar.

In [3]:

X = torch.randn(n, d_in)
# torch.randn(), standart normal dağılımdan rastgele sayılar üretir.
#
# X'in shape'i: (5, 4)
#               ↑  ↑
#               |  └── Her token için 4 özellik
#               └───── 5 token / zaman adımı
#
# X[0] -> 1. token'ın 4 boyutlu vektörü
# X[1] -> 2. token'ın 4 boyutlu vektörü
# X[2] -> 3. token'ın 4 boyutlu vektörü
# X[3] -> 4. token'ın 4 boyutlu vektörü
# X[4] -> 5. token'ın 4 boyutlu vektörü
#
# Buradaki sayılar gerçek kelimelerin embedding'leri değil,
# RNN'in çalışma mantığını göstermek için üretilmiş rastgele değerlerdir.
#
# Örneğin X[0], "The" token'ının temsilini düşünmemize yardımcı olur.


# ---------------------------------------------------------
# 3. GİRDİ AĞIRLIKLARINI OLUŞTURMA
# ---------------------------------------------------------

Wx = torch.randn(d_in, d_h)
# Wx = input-to-hidden ağırlık matrisi.
#
# Shape: (4, 3)
#
# Her zaman adımındaki girdi vektörünü alır ve gizli durum boyutuna
# dönüştürülmesine katkıda bulunur.
#
# X[t] shape'i:       (4,)
# Wx shape'i:         (4, 3)
#
# X[t] @ Wx işlemi:
# (4,) @ (4, 3) -> (3,)
#
# Sonuç, gizli durumun 3 bileşenine katkı sağlayan bir vektördür.
#
# Not: Bu örnekte Wx rastgele oluşturuluyor ve eğitim yapılmıyor.


# ---------------------------------------------------------
# 4. ÖNCEKİ GİZLİ DURUMUN AĞIRLIKLARINI OLUŞTURMA
# ---------------------------------------------------------

Wh = torch.randn(d_h, d_h)
# Wh = hidden-to-hidden ağırlık matrisi.
#
# Shape: (3, 3)
#
# Önceki zaman adımındaki gizli durumu alır ve yeni gizli duruma
# katkı sağlayacak şekilde dönüştürür.
#
# h shape'i:          (3,)
# Wh shape'i:         (3, 3)
#
# h @ Wh işlemi:
# (3,) @ (3, 3) -> (3,)
#
# RNN'in geçmiş bilgiyi sonraki adıma taşımasını sağlayan temel
# mekanizmalardan biri budur.
#
# Önemli: Wh, her zaman adımında yeniden oluşturulmaz.
# Aynı Wh matrisi bütün zaman adımlarında paylaşılır.


# ---------------------------------------------------------
# 5. BIAS VE İLK GİZLİ DURUM
# ---------------------------------------------------------

b = torch.zeros(d_h)
# Bias vektörü oluşturuyoruz.
#
# Shape: (3,)
# Başlangıçta bütün bileşenleri sıfır:
# b = [0, 0, 0]
#
# Bias, doğrusal dönüşümün sonucuna eklenen öğrenilebilir bir parametredir.
# Bu örnekte sıfır başlatılmıştır ve eğitim yapılmadığı için değişmez.


h = torch.zeros(d_h)
# RNN'in başlangıç gizli durumunu, yani h_0'ı oluşturuyoruz.
#
# Shape: (3,)
# h_0 = [0, 0, 0]
#
# Henüz hiçbir token işlenmediği için başlangıçta geçmiş bilgi yok.
# Bu nedenle başlangıç gizli durumunu sıfır seçiyoruz.
#
# Her yeni token işlendiğinde h değişkeni güncellenecek.
# Yeni h, hem mevcut token'dan hem de önceki h'den etkilenecek.


# ---------------------------------------------------------
# 6. GİZLİ DURUMLARI SAKLAYACAĞIMIZ LİSTE
# ---------------------------------------------------------

states = []
# Her zaman adımında elde edilen gizli durumu burada saklayacağız.
#
# 1. adım -> h_1
# 2. adım -> h_2
# 3. adım -> h_3
# 4. adım -> h_4
# 5. adım -> h_5
#
# Neden saklıyoruz?
# Çünkü yalnızca son gizli durumu değil, dizinin bütün ara durumlarını
# da görmek istiyoruz.

In [4]:
# ---------------------------------------------------------
# 7. RNN'İN SIRALI ÇALIŞMASI
# ---------------------------------------------------------

for t in range(n):
    # t sırasıyla 0, 1, 2, 3, 4 değerlerini alır.
    #
    # Her iterasyonda diziden bir token işlenir:
    # t = 0 -> X[0], yani ilk token
    # t = 1 -> X[1], yani ikinci token
    # t = 2 -> X[2], yani üçüncü token
    # t = 3 -> X[3], yani dördüncü token
    # t = 4 -> X[4], yani beşinci token
    #
    # RNN burada sıralı çalışır; çünkü yeni gizli durum,
    # bir önceki zaman adımında hesaplanan h değerine bağlıdır.


    # -----------------------------------------------------
    # 7.1. YENİ GİZLİ DURUMU HESAPLAMA
    # -----------------------------------------------------

    h = torch.tanh(X[t] @ Wx + h @ Wh + b)
    # RNN'in temel güncelleme denklemi:
    #
    # h_t = tanh(x_t @ Wx + h_(t-1) @ Wh + b)
    #
    # Burada:
    # x_t       -> Şu an işlenen token'ın girdi vektörü
    # h_(t-1)   -> Önceki zaman adımının gizli durumu
    # Wx        -> Girdi ağırlıkları
    # Wh        -> Gizli durum ağırlıkları
    # b         -> Bias vektörü
    # tanh      -> Sonucu -1 ile +1 arasına sıkıştıran aktivasyon fonksiyonu
    #
    # İŞLEMLERİ TEK TEK İNCELEYELİM:
    #
    # A) X[t] @ Wx
    #    (4,) @ (4, 3) -> (3,)
    #    Mevcut token'ın gizli duruma katkısını hesaplar.
    #
    # B) h @ Wh
    #    (3,) @ (3, 3) -> (3,)
    #    Önceki gizli durumun yeni duruma katkısını hesaplar.
    #
    # C) X[t] @ Wx + h @ Wh + b
    #    (3,) + (3,) + (3,) -> (3,)
    #    İki katkıyı ve bias'ı toplar.
    #
    # D) torch.tanh(...)
    #    (3,) -> (3,)
    #    Her bileşene tanh uygulanır.
    #
    # ÇOK ÖNEMLİ:
    # Sağ taraftaki h, güncellemeden önce önceki gizli durumdur.
    # Atama tamamlandıktan sonra h artık yeni gizli durumu tutar.
    #
    # Böylece sonraki iterasyonda hesaplama yeni h üzerinden devam eder.


    # -----------------------------------------------------
    # 7.2. BU ZAMAN ADIMININ DURUMUNU SAKLAMA
    # -----------------------------------------------------

    states.append(h)
    # Hesaplanan gizli durumu listeye ekliyoruz.
    #
    # İlk iterasyon sonunda: states = [h_1]
    # İkinci iterasyon sonunda: states = [h_1, h_2]
    # ...
    # Son iterasyon sonunda: states = [h_1, h_2, h_3, h_4, h_5]
    #
    # Her h'nin shape'i (3,)'tür.

In [5]:
# ---------------------------------------------------------
# 8. GİZLİ DURUMLARI TEK BİR TENSÖRDE BİRLEŞTİRME
# ---------------------------------------------------------

H = torch.stack(states)
# torch.stack(), listedeki aynı boyutlu tensörleri yeni bir eksen
# oluşturarak birleştirir.
#
# states listesinde 5 adet gizli durum var.
# Her birinin shape'i: (3,)
#
# Birleştirme:
# 5 adet (3,) -> (5, 3)
#
# H'nin satırları zaman adımlarını, sütunları gizli durum bileşenlerini
# temsil eder:
#
# H[0] -> h_1, ilk token işlendiğinde oluşan durum
# H[1] -> h_2, ikinci token işlendiğinde oluşan durum
# H[2] -> h_3, üçüncü token işlendiğinde oluşan durum
# H[3] -> h_4, dördüncü token işlendiğinde oluşan durum
# H[4] -> h_5, beşinci token işlendiğinde oluşan durum
#
# Dolayısıyla H'nin shape'i (5, 3)'tür.


# ---------------------------------------------------------
# 9. BOYUTLARI YAZDIRMA
# ---------------------------------------------------------

print("X:", X.shape, "| H:", H.shape)
# Beklenen çıktı:
# X: torch.Size([5, 4]) | H: torch.Size([5, 3])
#
# X: 5 token, token başına 4 girdi özelliği.
# H: 5 zaman adımı, zaman adımı başına 3 gizli durum bileşeni.

X: torch.Size([5, 4]) | H: torch.Size([5, 3])
